# Lab 06 extra: How MCP messages look on the wire

**Going further, for Lab 06**

## Goal

*On the wire* means the messages themselves, as they travel between a client and a server. In the two notebooks
of this lab a client spoke to the server for us. Here we start the shopping-list server
ourselves and write each message by hand, one line of JSON at a time. That shows what a client does on every call,
which two keys every request must carry, how the server refuses a request without them, and why a server must never
`print()` to its standard output.

## What you will learn

- How to start an MCP server as a process and exchange JSON-RPC messages with it, one line each.
- What the request envelope is: the two `_meta` keys that every request of the 2026-07-28 revision carries.
- How to tell apart the two errors a request without the envelope gets, `-32602` and `-32600`, and a tool that
  failed, which is not a protocol error.
- Why a stdio server's standard output belongs to the protocol, and where its diagnostics go instead.

## Setup

This notebook calls no model, so it needs no key. The first cell finds the repository folder and the server file.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

SERVER_FILE = ROOT / "06_MCP" / "shopping_server.py"
LIST_FILE = ROOT / "06_MCP" / "outputs" / "shopping_list.txt"

# Start from a known list: two items, one per line
LIST_FILE.parent.mkdir(exist_ok=True)
LIST_FILE.write_text("milk\neggs\n", encoding="utf-8")

print(SERVER_FILE.name, "|", LIST_FILE.read_text(encoding="utf-8").splitlines())

shopping_server.py | ['milk', 'eggs']


`shopping_server.py` is the server of notebook 1: two tools, `add_item` and `list_items`, over a text file with
one item per line. The cell also wrote a fresh list with milk and eggs, so the outputs below are the same on every
run. (Notebook 1 uses the same file and resets it when it starts.)

---

## Part A: Write a request by hand and read the reply

A client such as `mcp.Client` hides the messages. To see them, we do the client's work ourselves.

### Start the server as a process

`subprocess.Popen` starts a program and leaves it running. We start the server with this notebook's Python
(`sys.executable`) and ask for its three standard streams as pipes:

- `stdin`: what the server reads. We write our requests here.
- `stdout`: what the server writes. We read its replies here.
- `stderr`: a separate stream for diagnostics. Part C reads it.

`text=True` with `encoding="utf-8"` lets us write and read strings instead of bytes.

In [2]:
process = subprocess.Popen(
    [sys.executable, str(SERVER_FILE)],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    encoding="utf-8",
)

print("server running:", process.poll() is None)

server running: True


`process.poll()` returns `None` while a process is alive. The server is running and waiting for a line on its
`stdin`.

### A request is one line of JSON

Every MCP message is a **JSON-RPC 2.0** message. A request is a JSON object with four keys:

| Key | What it holds |
|---|---|
| `jsonrpc` | always `"2.0"` |
| `id` | a number we choose; the reply carries the same one |
| `method` | what we ask for, such as `tools/list` |
| `params` | the details of the request |

**The envelope.** In the 2026-07-28 revision of MCP, every request says for itself which revision of the
protocol it speaks and what the client can do. Both go in `params`, under `_meta`, as two keys with long names.
Our client offers nothing extra, so its capabilities are an empty object.

In [3]:
META = {
    "io.modelcontextprotocol/protocolVersion": "2026-07-28",
    "io.modelcontextprotocol/clientCapabilities": {},
}

request = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {"_meta": META}}

print(json.dumps(request))

{"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {"_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28", "io.modelcontextprotocol/clientCapabilities": {}}}}


`json.dumps` writes the dictionary as one line of JSON. That line is the whole request: on stdio a message is one
line, ended by a line break.

We write the line to the server's `stdin`. `flush()` pushes it through the pipe at once, and `readline()` waits
for one line to come back.

In [4]:
process.stdin.write(json.dumps(request) + "\n")
process.stdin.flush()

line = process.stdout.readline()
print(line)

{"jsonrpc":"2.0","id":1,"result":{"cacheScope":"private","resultType":"complete","tools":[{"description":"Add one item to the shopping list.","inputSchema":{"type":"object","properties":{"item":{"title":"Item","type":"string"}},"required":["item"],"title":"add_itemArguments"},"name":"add_item","outputSchema":{"properties":{"result":{"title":"Result","type":"string"}},"required":["result"],"title":"add_itemOutput","type":"object"}},{"description":"List every item on the shopping list, in the order they were added.","inputSchema":{"type":"object","properties":{},"title":"list_itemsArguments"},"name":"list_items","outputSchema":{"properties":{"result":{"items":{"type":"string"},"title":"Result","type":"array"}},"required":["result"],"title":"list_itemsOutput","type":"object"}}],"ttlMs":0,"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"shopping","version":""}}}}



One line of JSON came back. It is long, so we parse it with `json.loads` and read it piece by piece.

In [5]:
reply = json.loads(line)

print("keys of the reply: ", list(reply))
print("id:                ", reply["id"])
print("keys of the result:", list(reply["result"]))
print("tools:             ", [tool["name"] for tool in reply["result"]["tools"]])

keys of the reply:  ['jsonrpc', 'id', 'result']
id:                 1
keys of the result: ['cacheScope', 'resultType', 'tools', 'ttlMs', '_meta']
tools:              ['add_item', 'list_items']


- The reply has our `id`, 1, so we know which request it answers, and a `result`.
- `result["tools"]` lists `add_item` and `list_items`. This is what `list_tools()` returns in the notebooks.
- `resultType` says whether the result is final (`complete`). `cacheScope` and `ttlMs` tell a client whether
  it may keep the answer and for how long. `_meta` carries the server's name.

One tool in full, as it travels:

In [6]:
print(json.dumps(reply["result"]["tools"][0], indent=2))

{
  "description": "Add one item to the shopping list.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "item": {
        "title": "Item",
        "type": "string"
      }
    },
    "required": [
      "item"
    ],
    "title": "add_itemArguments"
  },
  "name": "add_item",
  "outputSchema": {
    "properties": {
      "result": {
        "title": "Result",
        "type": "string"
      }
    },
    "required": [
      "result"
    ],
    "title": "add_itemOutput",
    "type": "object"
  }
}


The description is the function's docstring, `inputSchema` describes the arguments and `outputSchema` the result.
In the JSON the names are in camel case (`inputSchema`); the Python client shows the same fields as
`input_schema` and `output_schema`.

### Call a tool by hand

The three steps (write a line, flush, read a line) are the same for every message, so we put them in a small
function, `send`, which also parses the reply.

A `tools/call` request names the tool and its arguments in `params`, beside the envelope.

In [7]:
def send(process, message):
    """Write one JSON-RPC message as a line, read one line back and parse it."""
    process.stdin.write(json.dumps(message) + "\n")
    process.stdin.flush()
    return json.loads(process.stdout.readline())


call = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {"name": "add_item", "arguments": {"item": "bread"}, "_meta": META},
}

reply = send(process, call)
print(json.dumps(reply["result"], indent=2))

{
  "content": [
    {
      "text": "Added bread.",
      "type": "text"
    }
  ],
  "isError": false,
  "resultType": "complete",
  "structuredContent": {
    "result": "Added bread."
  },
  "_meta": {
    "io.modelcontextprotocol/serverInfo": {
      "name": "shopping",
      "version": ""
    }
  }
}


The result of a tool call, key by key:

- `content`: a list with one text block, `Added bread.` This is the text a host gives to the model.
- `structuredContent`: the same result as data, `{"result": "Added bread."}`.
- `isError`: `false`. The tool ran and returned.

The file confirms it. The server process wrote the line; we sent only text down a pipe.

In [8]:
print(LIST_FILE.read_text(encoding="utf-8").splitlines())

['milk', 'eggs', 'bread']


**Checkpoint A.** The server answers `tools/list` with `add_item` and `list_items`, under the `id` we sent.
`tools/call` for `add_item` with `bread` returns `Added bread.` with `isError` `false`, and the file holds milk,
eggs and bread.

---

## Part B: Tell apart the errors a request without the envelope gets

Many examples of MCP on the web were written before the 2026-07-28 revision. They send `tools/list` with empty
`params`. This part sends that request twice, in two situations, and reads the two different errors it leads to.
Knowing them saves time when a client and a server disagree about the revision.

### On a connection that already speaks 2026-07-28

Our connection has carried two requests with the envelope. Now the same `tools/list`, without it.

In [9]:
bare = {"jsonrpc": "2.0", "id": 3, "method": "tools/list", "params": {}}

print(json.dumps(send(process, bare), indent=2))

{
  "jsonrpc": "2.0",
  "id": 3,
  "error": {
    "code": -32602,
    "message": "params._meta must be an object carrying the required 'io.modelcontextprotocol/protocolVersion' and 'io.modelcontextprotocol/clientCapabilities' envelope keys"
  }
}


The reply has an `error` where a `result` would be:

- **`code` -32602** is JSON-RPC's code for invalid parameters: the method exists, and what we sent with it is
  wrong.
- **`message`** says what is missing: `params._meta` must carry the two envelope keys.

The request is valid JSON, and `tools/list` is a real method. What it lacks is the envelope. What came earlier on
the connection does not count: each request must name its revision itself.

We stop this server by closing its `stdin`. A stdio server ends when its input ends. `wait()` returns the exit
code.

In [10]:
process.stdin.close()
print("exit code:", process.wait())

exit code: 0


Exit code 0: a clean stop.

### On a fresh connection

What if the request without the envelope is the first one a server receives? We start a new process, send the bare
request first and the enveloped one after it.

In [11]:
process = subprocess.Popen(
    [sys.executable, str(SERVER_FILE)],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    encoding="utf-8",
)

print(send(process, bare)["error"])
print(send(process, request)["error"])

{'code': -32602, 'message': 'Invalid request parameters', 'data': ''}
{'code': -32600, 'message': 'this connection serves the handshake protocol era; requests carrying the 2026-07-28 envelope are not accepted on it'}


Two errors, and neither is the one we got above:

1. The bare request gets -32602 again, with a shorter message: `Invalid request parameters`. No word about the
   envelope.
2. The request **with** the envelope, which the first process answered in Part A, is now refused with **-32600**,
   JSON-RPC's code for an invalid request: *this connection serves the handshake protocol era*.

**The first request decides which revision a connection speaks.** A first request with the envelope opens a
2026-07-28 connection. Anything else is taken for a client of the older revisions, the **handshake era**, in which
a connection had to open with a request called `initialize`. A `tools/list` before that handshake is invalid,
which is error 1. And once a connection is in the handshake era, it stays there, which is error 2.

The server still serves the older revision to clients that ask for it. We finish the handshake on this
connection: an `initialize` request, a notification that the client is ready (a message with no `id`, which gets no
reply), and then the bare `tools/list` again.

In [12]:
handshake = {
    "jsonrpc": "2.0",
    "id": 4,
    "method": "initialize",
    "params": {"protocolVersion": "2025-11-25", "capabilities": {}, "clientInfo": {"name": "by-hand", "version": "1"}},
}
print("protocolVersion:", send(process, handshake)["result"]["protocolVersion"])

# A notification has no id, and the server sends no reply to it
process.stdin.write(json.dumps({"jsonrpc": "2.0", "method": "notifications/initialized"}) + "\n")
process.stdin.flush()

reply = send(process, {"jsonrpc": "2.0", "id": 5, "method": "tools/list", "params": {}})
print("tools:", [tool["name"] for tool in reply["result"]["tools"]])

protocolVersion: 2025-11-25
tools: ['add_item', 'list_items']


The server agrees on `2025-11-25`, the last revision with a handshake, and after it the bare `tools/list` is
answered.

This is what the MCP Inspector does by default, which is why its page shows `MCP 2025-11-25` beside our server in
notebook 1. The Python client sends the envelope and gets `2026-07-28`. One server file, two revisions, and each
connection speaks one of them.

### A tool that fails is a result, not a protocol error

One more case, to keep it apart from the two above. We call `add_item` without the `item` it requires. This
connection speaks the older revision now, so the request carries no envelope.

In [13]:
broken_call = {"jsonrpc": "2.0", "id": 6, "method": "tools/call", "params": {"name": "add_item", "arguments": {}}}

reply = send(process, broken_call)
print("keys of the reply:", list(reply))
print("isError:", reply["result"]["isError"])
print(reply["result"]["content"][0]["text"])

process.stdin.close()
print("exit code:", process.wait())

keys of the reply: ['jsonrpc', 'id', 'result']
isError: True
Error executing tool add_item: 1 validation error for add_itemArguments
item
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
exit code: 0


The reply has a `result`, and no `error`. The message itself was fine. The tool is what failed: the server checked
the arguments against the tool's schema, found `item` missing (`Field required`) and ran nothing. It reports
that with `isError` `true` and a text that says what was wrong.

| What went wrong | What comes back | Who reads it |
|---|---|---|
| the message (no envelope, wrong revision, unknown method) | an `error` with a code such as -32602 or -32600 | the client's programmer |
| the tool (bad arguments, or the tool raised an exception) | a normal `result` with `isError: true` and a text | the host sends the text to the model, which can try again |

**Checkpoint B.** On the first connection the bare `tools/list` gets -32602 and a message that names the two
envelope keys. On a fresh connection it gets -32602 `Invalid request parameters`, and the enveloped request after
it gets -32600. After `initialize` the server reports `2025-11-25` and answers the bare `tools/list`. `add_item`
without an `item` returns a result with `isError` `True` and `Field required`.

---

## Part C: Keep the server's standard output for the protocol

On stdio, the server's standard output *is* the channel: every line a client reads there must be a JSON-RPC
message. That makes a harmless habit a problem. A `print()` inside a tool, added to see what the tool is doing,
writes to the same channel.

### A server that prints

We write a small server file with one tool that prints before it returns. It goes in `outputs/`, a folder git
ignores.

In [14]:
NOISY_FILE = ROOT / "06_MCP" / "outputs" / "noisy_server.py"

NOISY_FILE.write_text('''from mcp.server import MCPServer

mcp = MCPServer("noisy")


@mcp.tool()
def say_hello(name: str) -> str:
    """Greet someone by name."""
    print(f"greeting {name}")
    return f"Hello, {name}!"


if __name__ == "__main__":
    mcp.run("stdio")
''', encoding="utf-8")

print(NOISY_FILE.read_text(encoding="utf-8"))

from mcp.server import MCPServer

mcp = MCPServer("noisy")


@mcp.tool()
def say_hello(name: str) -> str:
    """Greet someone by name."""
    print(f"greeting {name}")
    return f"Hello, {name}!"


if __name__ == "__main__":
    mcp.run("stdio")



We start it, call `say_hello`, close its input and wait for it to stop. Then we read everything else the server
wrote: `process.stdout.read()` returns what is left on its standard output, and `process.stderr.read()` what it
wrote to its error stream. `repr` shows the line breaks.

In [15]:
hello = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {"name": "say_hello", "arguments": {"name": "Sam"}, "_meta": META},
}

process = subprocess.Popen(
    [sys.executable, str(NOISY_FILE)],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    encoding="utf-8",
)
reply = send(process, hello)
process.stdin.close()
process.wait()

print("reply:              ", reply["result"]["content"][0]["text"])
print("more on stdout:     ", repr(process.stdout.read()))
print("on the error stream:", repr(process.stderr.read()))

reply:               Hello, Sam!
more on stdout:      'greeting Sam\n'
on the error stream: ''


The reply is right: `Hello, Sam!` And after it, the server's standard output carried one more line,
`greeting Sam`. That line is not a JSON-RPC message, and it sits on the channel where a client expects only
messages.

In our run the stray line reached the standard output only when the server stopped, after the reply, so our
`readline()` never read it. It came last because of two things we did not choose: Python held the printed text in a buffer and wrote it out only when the server stopped. While it serves, the `mcp`
package points the standard output at the error stream, and it points it back when it stops. So the late text landed
on the real standard output, after the reply, and the error stream got nothing. That timing is an accident of this
server, not a promise of the protocol: a server built another way can put the line in the middle of a conversation,
and the official Python client then logs `Failed to parse JSONRPC message from server` for it. The protocol's rule is plain: a stdio server **must
not** write anything to its standard output that is not a message.

### Diagnostics go to the error stream

The fix is one argument: `print(..., file=sys.stderr)`. The error stream is not part of the protocol, and a client
may show it, save it or ignore it. Python's `logging` module writes there by default as well.

`str.replace` makes the two changes in the file's text: the `print` gets `file=sys.stderr`, and the file gets the
`import sys` that this needs.

In [16]:
QUIET_FILE = ROOT / "06_MCP" / "outputs" / "quiet_server.py"

fixed = NOISY_FILE.read_text(encoding="utf-8")
fixed = fixed.replace('print(f"greeting {name}")', 'print(f"greeting {name}", file=sys.stderr)')
fixed = "import sys\n\n" + fixed
QUIET_FILE.write_text(fixed, encoding="utf-8")

process = subprocess.Popen(
    [sys.executable, str(QUIET_FILE)],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    encoding="utf-8",
)
reply = send(process, hello)
process.stdin.close()
process.wait()

print("reply:              ", reply["result"]["content"][0]["text"])
print("more on stdout:     ", repr(process.stdout.read()))
print("on the error stream:", repr(process.stderr.read()))

reply:               Hello, Sam!
more on stdout:      ''
on the error stream: 'greeting Sam\n'


The same reply, nothing more on the standard output, and `greeting Sam` on the error stream, where a diagnostic
belongs.

**Checkpoint C.** With `print()`: the reply is `Hello, Sam!`, and `'greeting Sam\n'` follows on the standard
output. With `print(..., file=sys.stderr)`: the standard output holds nothing more, and the error stream holds
`'greeting Sam\n'`.

---

## What you can do now

- Start an MCP server as a process and exchange JSON-RPC messages with it by hand (Part A).
- Write the 2026-07-28 envelope, and say what each of its two keys tells the server (Part A).
- Read a -32602 and a -32600 error, say which request caused each, and tell both from a tool result with
  `isError` `true` (Part B).
- Keep a stdio server's standard output for the protocol and send diagnostics to the error stream (Part C).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| -32602 `params._meta must be an object carrying the required ... envelope keys` | a request without the envelope, on a 2026-07-28 connection | add `"_meta": META` to `params` |
| -32602 `Invalid request parameters` on the first request | a first request without the envelope puts the connection in the handshake era, where `tools/list` must come after `initialize` | send the envelope on the first request, or do the handshake |
| -32600 `this connection serves the handshake protocol era` | a request with the envelope, on a connection whose first request had none | start a fresh server process and send the enveloped request first |
| a cell that calls `send` never finishes | the message had no `id` (a notification), so the server sent no reply and `readline()` keeps waiting | interrupt the kernel; send a notification with `write` and `flush` only, as Part B does |
| the client's log says `Failed to parse JSONRPC message from server` | the server wrote something that is not a message to its standard output, often a `print()` | `print(..., file=sys.stderr)` or `logging` in the server |

## Resources

- [MCP specification 2026-07-28: key changes](https://modelcontextprotocol.io/specification/2026-07-28/changelog):
  what the revision removed, added and deprecated
- [MCP specification: the stdio transport](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/stdio):
  one message per line, the standard output only for messages
- [MCP: versioning](https://modelcontextprotocol.io/docs/2026-07-28/learn/versioning): revisions, and how old and
  new clients and servers meet
- [MCP Inspector: protocol eras](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector/protocol-eras):
  how the Inspector chooses between the handshake and the envelope
- [JSON-RPC 2.0 specification](https://www.jsonrpc.org/specification): requests, responses, notifications and
  error codes
- [Python: `subprocess.Popen`](https://docs.python.org/3/library/subprocess.html#popen-constructor)

Back to [notebook 1](06a_meet_mcp.ipynb) or [notebook 2](06b_mcp_at_the_desk.ipynb).